# 03 - Evaluation: threshold, metrics, and the Isolation Forest baseline
The anomaly score of a flow is its reconstruction error. The threshold is chosen from
**validation NORMAL** errors only; the test set is used only to measure the result.

In [1]:
import sys; sys.path.append("..")
import json, numpy as np, torch, matplotlib.pyplot as plt, pandas as pd
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import IsolationForest
from src.model import Autoencoder

meta = json.load(open("../artifacts/metadata.json"))
X_train = np.load("../data/processed/X_train_normal.npy")
X_val = np.load("../data/processed/X_val_normal.npy")
X_test = np.load("../data/processed/X_test.npy")
y_test = np.load("../data/processed/y_test.npy")

model = Autoencoder(meta["input_dim"], meta["bottleneck"])
model.load_state_dict(torch.load("../artifacts/autoencoder.pth"))
model.eval()

def recon_error(X):
    with torch.no_grad():
        x = torch.from_numpy(X)
        return ((model(x) - x) ** 2).mean(dim=1).numpy()   # one score per flow

err_val, err_test = recon_error(X_val), recon_error(X_test)
print("mean error  normal(val):", err_val.mean().round(5),
      "| test normal:", err_test[y_test == 0].mean().round(5),
      "| test attack:", err_test[y_test == 1].mean().round(5))

mean error  normal(val): 0.00041 | test normal: 0.00099 | test attack: 0.01653


## Threshold choice: percentile of validation normal errors (trade-off, not a free lunch)

In [2]:
rows = []
for p in [90, 95, 97, 99]:
    thr = np.percentile(err_val, p)
    pred = (err_test > thr).astype(int)
    rows.append({"percentile": p, "threshold": thr,
                 "precision": precision_score(y_test, pred),
                 "recall": recall_score(y_test, pred), "f1": f1_score(y_test, pred)})
pd.DataFrame(rows).round(4)

,percentile,threshold,precision,recall,f1
0,90,0.0006,0.9049,0.9591,0.9312
1,95,0.0013,0.9173,0.8612,0.8884
2,97,0.0022,0.9262,0.8279,0.8743
3,99,0.0068,0.9734,0.6873,0.8057


In [3]:
PERCENTILE = 95          # final choice (chosen from validation errors only)
threshold = float(np.percentile(err_val, PERCENTILE))
pred = (err_test > threshold).astype(int)
auc_ae = roc_auc_score(y_test, err_test)
res_ae = dict(model="Autoencoder", precision=precision_score(y_test, pred),
              recall=recall_score(y_test, pred), f1=f1_score(y_test, pred), auc=auc_ae)
print("threshold:", threshold); print(res_ae)

threshold: 0.0013113709865137935
{'model': 'Autoencoder', 'precision': 0.9173306772908366, 'recall': 0.8612171744720643, 'f1': 0.8883887303564969, 'auc': 0.9628613457400121}


## Baseline: Isolation Forest (also trained on normal data only)

In [4]:
iso = IsolationForest(n_estimators=200, contamination="auto", random_state=42, n_jobs=-1)
iso.fit(X_train)
s_val, s_test = -iso.score_samples(X_val), -iso.score_samples(X_test)   # higher = more anomalous
thr_iso = np.percentile(s_val, PERCENTILE)
pred_iso = (s_test > thr_iso).astype(int)
res_if = dict(model="Isolation Forest", precision=precision_score(y_test, pred_iso),
              recall=recall_score(y_test, pred_iso), f1=f1_score(y_test, pred_iso),
              auc=roc_auc_score(y_test, s_test))
results = pd.DataFrame([res_ae, res_if]).round(4)
results

,model,precision,recall,f1,auc
0,Autoencoder,0.9173,0.8612,0.8884,0.9629
1,Isolation Forest,0.9726,0.6841,0.8032,0.9445


## Plots for the README

In [5]:
import os; os.makedirs("../figures", exist_ok=True)
plt.figure(figsize=(8, 4))
bins = np.linspace(0, np.percentile(err_test, 99), 80)
plt.hist(err_test[y_test == 0], bins=bins, alpha=0.6, label="normal")
plt.hist(err_test[y_test == 1], bins=bins, alpha=0.6, label="attack")
plt.axvline(threshold, color="k", ls="--", label="threshold")
plt.xlabel("reconstruction error"); plt.ylabel("flows"); plt.legend()
plt.savefig("../figures/error_histogram.png", dpi=120, bbox_inches="tight"); plt.show()

plt.figure(figsize=(5, 5))
for name, sc in [("Autoencoder", err_test), ("Isolation Forest", s_test)]:
    fpr, tpr, _ = roc_curve(y_test, sc)
    plt.plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, sc):.3f})")
plt.plot([0, 1], [0, 1], "k:"); plt.xlabel("false positive rate"); plt.ylabel("true positive rate")
plt.legend(); plt.savefig("../figures/roc_curve.png", dpi=120, bbox_inches="tight"); plt.show()

ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["normal", "attack"]).plot()
plt.savefig("../figures/confusion_matrix.png", dpi=120, bbox_inches="tight"); plt.show()

## Save the threshold and results (the API reads the threshold)

In [6]:
meta["threshold"] = threshold
meta["threshold_percentile"] = PERCENTILE
meta["results"] = results.to_dict(orient="records")
json.dump(meta, open("../artifacts/metadata.json", "w"), indent=2)
results.to_csv("../figures/results.csv", index=False)
print("saved threshold", threshold)

saved threshold 0.0013113709865137935
